# Olist E-Commerce Feature Engineering

This notebook creates analysis-ready features from the cleaned Olist datasets for SQL analysis and Power BI reporting.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
PROCESSED_PATH = Path("../data/processed/")

In [3]:
orders = pd.read_csv(
    PROCESSED_PATH / "orders_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

In [4]:
order_items = pd.read_csv(
    PROCESSED_PATH / "order_items_clean.csv",
    parse_dates=[
        "shipping_limit_date"
    ]
)

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [6]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [7]:
orders["delivery_time_days"] = (
    (
        orders["order_delivered_customer_date"] -
        orders["order_purchase_timestamp"]
    )
    .dt.total_seconds()
    .div(86400)
    .round(2)
)

In [8]:
orders["delivery_delay_days"] = (
    (
        orders["order_delivered_customer_date"] -
        orders["order_estimated_delivery_date"]
    )
    .dt.total_seconds()
    .div(86400)
    .round(2)
)

In [9]:
orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_time_days",
        "delivery_delay_days"
    ]
].head(10)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.44,-7.11
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.78,-5.36
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.39,-17.25
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.21,-12.98
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.87,-9.24
5,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,16.54,-5.54
6,2017-04-11 12:22:08,NaT,2017-05-09,NaN,NaN
7,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,9.99,-11.46
8,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,9.82,-31.41
9,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,18.22,-6.28


In [10]:
orders[
    [
        "delivery_time_days",
        "delivery_delay_days"
    ]
].describe()

,delivery_time_days,delivery_delay_days
count,96476.000000,96476.000000
mean,12.558687,-11.179116
std,9.546528,10.186115
min,0.530000,-146.020000
25%,6.770000,-16.240000
50%,10.220000,-11.950000
75%,15.720000,-6.390000
max,209.630000,188.980000


In [11]:
actual_delivery_date = (
    orders["order_delivered_customer_date"]
    .dt.normalize()
)

estimated_delivery_date = (
    orders["order_estimated_delivery_date"]
    .dt.normalize()
)

In [12]:
is_delivered = orders["order_status"].eq("delivered")

In [13]:
orders["delivery_performance"] = np.select(
    [
        is_delivered &
        actual_delivery_date.notna() &
        (actual_delivery_date <= estimated_delivery_date),

        is_delivered &
        actual_delivery_date.notna() &
        (actual_delivery_date > estimated_delivery_date),

        is_delivered &
        actual_delivery_date.isna()
    ],
    [
        "On Time",
        "Late",
        "Missing Delivery Date"
    ],
    default="Not Delivered"
)

In [14]:
orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_time_days",
        "delivery_delay_days",
        "delivery_performance"
    ]
].head(10)

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,delivery_performance
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,On Time
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,On Time
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,On Time
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,On Time
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,On Time
5,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,16.54,-5.54,On Time
6,2017-04-11 12:22:08,NaT,2017-05-09,NaN,NaN,Not Delivered
7,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,9.99,-11.46,On Time
8,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,9.82,-31.41,On Time
9,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,18.22,-6.28,On Time


In [15]:
orders[
    "delivery_performance"
].value_counts()

delivery_performance
On Time                  89936
Late                      6534
Not Delivered             2963
Missing Delivery Date        8
Name: count, dtype: int64

In [16]:
pd.crosstab(
    orders["order_status"],
    orders["delivery_performance"]
)

delivery_performance,Late,Missing Delivery Date,Not Delivered,On Time
order_status,,,,
approved,0,0,2,0
canceled,0,0,625,0
created,0,0,5,0
delivered,6534,8,0,89936
invoiced,0,0,314,0
processing,0,0,301,0
shipped,0,0,1107,0
unavailable,0,0,609,0


### Orders Feature Engineering

- Calculated the actual delivery time in days from purchase to customer delivery.
- Calculated delivery delay as the difference between actual and estimated delivery dates.
- Classified delivered orders as `On Time`, `Late`, or `Missing Delivery Date`.
- Non-delivered order statuses were classified as `Not Delivered`.
- Delivery performance was limited to completed deliveries to avoid including canceled or incomplete orders in delivery KPIs.

# Calculating Item Total

In [17]:
order_items["item_total"] = (
    order_items["price"] +
    order_items["freight_value"]
).round(2)

In [18]:
order_items[
    [
        "order_id",
        "order_item_id",
        "price",
        "freight_value",
        "item_total"
    ]
].head(10)

,order_id,order_item_id,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,34.59
6,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,31.75
7,000576fe39319847cbb9d288c5617fa6,1,810.00,70.75,880.75
8,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,157.60
9,0005f50442cb953dcd1d21e1fb923495,1,53.99,11.40,65.39


In [19]:
order_items[
    [
        "price",
        "freight_value",
        "item_total"
    ]
].describe()

,price,freight_value,item_total
count,112650.000000,112650.000000,112650.000000
mean,120.653739,19.990320,140.644059
std,183.633928,15.806405,190.724394
min,0.850000,0.000000,6.080000
25%,39.900000,13.080000,55.220000
50%,74.990000,16.260000,92.320000
75%,134.900000,21.150000,157.937500
max,6735.000000,409.680000,6929.310000


In [20]:
order_item_summary = (
    order_items
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        item_count=(
            "order_item_id",
            "count"
        ),
        product_value=(
            "price",
            "sum"
        ),
        freight_value=(
            "freight_value",
            "sum"
        ),
        order_total_value=(
            "item_total",
            "sum"
        )
    )
)

In [21]:
money_columns = [
    "product_value",
    "freight_value",
    "order_total_value"
]

order_item_summary[money_columns] = (
    order_item_summary[money_columns]
    .round(2)
)

In [22]:
order_item_summary.head()

,order_id,item_count,product_value,freight_value,order_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04


In [23]:
order_item_summary.shape

(98666, 5)

In [24]:
order_item_summary["order_id"].duplicated().sum()

np.int64(0)

In [25]:
orders.shape

(99441, 11)

In [26]:
orders_without_items = orders[
    ~orders["order_id"].isin(
        order_item_summary["order_id"]
    )
]

In [27]:
orders_without_items.shape

(775, 11)

In [28]:
orders_without_items[
    "order_status"
].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [29]:
orders_enriched = orders.merge(
    order_item_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [30]:
orders_enriched["has_item_data"] = (
    orders_enriched["item_count"].notna()
)

In [31]:
orders_enriched["item_count"] = (
    orders_enriched["item_count"]
    .fillna(0)
    .astype("int64")
)

In [32]:
orders_enriched.shape

(99441, 16)

In [33]:
orders_enriched[
    [
        "order_id",
        "order_status",
        "item_count",
        "product_value",
        "freight_value",
        "order_total_value",
        "has_item_data"
    ]
].head()

,order_id,order_status,item_count,product_value,freight_value,order_total_value,has_item_data
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,1,29.99,8.72,38.71,True
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,1,118.70,22.76,141.46,True
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,1,159.90,19.22,179.12,True
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,1,45.00,27.20,72.20,True
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,1,19.90,8.72,28.62,True


In [34]:
orders_enriched.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,delivery_performance,item_count,product_value,freight_value,order_total_value,has_item_data
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,On Time,1,29.99,8.72,38.71,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,On Time,1,118.70,22.76,141.46,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,On Time,1,159.90,19.22,179.12,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,On Time,1,45.00,27.20,72.20,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,On Time,1,19.90,8.72,28.62,True


# Creatin Payment Summary

In [35]:
payments = pd.read_csv(
    PROCESSED_PATH / "payments_clean.csv"
)

In [36]:
payment_summary = (
    payments
    .groupby(
        "order_id",
        as_index=False
    )
    .agg(
        total_payment=(
            "payment_value",
            "sum"
        ),
        payment_count=(
            "payment_sequential",
            "count"
        ),
        payment_method_count=(
            "payment_type",
            "nunique"
        ),
        max_installments=(
            "payment_installments",
            "max"
        )
    )
)

In [37]:
payment_summary["total_payment"] = (
    payment_summary["total_payment"]
    .round(2)
)

In [38]:
payment_summary.head()

,order_id,total_payment,payment_count,payment_method_count,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3


In [39]:
payment_summary.shape

(99437, 5)

In [40]:
payment_summary["order_id"].duplicated().sum()

np.int64(0)

In [41]:
orders_without_payments = orders_enriched[
    ~orders_enriched["order_id"].isin(
        payment_summary["order_id"]
    )
]

In [42]:
orders_without_payments[
    [
        "order_id",
        "order_status",
        "item_count",
        "order_total_value"
    ]
]

,order_id,order_status,item_count,order_total_value
1130,00b1cb0320190ca0daa2c88b35206009,canceled,0,NaN
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,3,143.46
39919,4637ca194b6387e2d538dc89b124b0ee,canceled,0,NaN
40235,c8c528189310eaa44a745b8d9d26908b,canceled,0,NaN


In [43]:
orders_without_payments[
    "order_status"
].value_counts()

order_status
canceled     3
delivered    1
Name: count, dtype: int64

In [44]:
orders_enriched = orders_enriched.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [45]:
orders_enriched["has_payment_data"] = (
    orders_enriched["total_payment"].notna()
)

In [46]:
payment_count_columns = [
    "payment_count",
    "payment_method_count",
    "max_installments"
]

orders_enriched[payment_count_columns] = (
    orders_enriched[payment_count_columns]
    .fillna(0)
    .astype("int64")
)

In [47]:
orders_enriched.loc[
    (~orders_enriched["has_payment_data"]) &
    (orders_enriched["order_status"] == "canceled"),
    "total_payment"
] = 0


In [48]:
orders_enriched.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,...,item_count,product_value,freight_value,order_total_value,has_item_data,total_payment,payment_count,payment_method_count,max_installments,has_payment_data
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,1,29.99,8.72,38.71,True,38.71,3,2,1,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,1,118.70,22.76,141.46,True,141.46,1,1,1,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,1,159.90,19.22,179.12,True,179.12,1,1,3,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,1,45.00,27.20,72.20,True,72.20,1,1,1,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,1,19.90,8.72,28.62,True,28.62,1,1,1,True


In [49]:
orders_enriched["payment_difference"] = (
    orders_enriched["total_payment"] -
    orders_enriched["order_total_value"]
).round(2)

In [50]:
orders_enriched.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,...,product_value,freight_value,order_total_value,has_item_data,total_payment,payment_count,payment_method_count,max_installments,has_payment_data,payment_difference
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,29.99,8.72,38.71,True,38.71,3,2,1,True,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,118.70,22.76,141.46,True,141.46,1,1,1,True,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,159.90,19.22,179.12,True,179.12,1,1,3,True,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,45.00,27.20,72.20,True,72.20,1,1,1,True,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,19.90,8.72,28.62,True,28.62,1,1,1,True,0.0


In [51]:
orders_enriched[
    orders_enriched["payment_difference"].abs() > 0.01
][
    [
        "order_id",
        "order_status",
        "order_total_value",
        "total_payment",
        "payment_difference"
    ]
].head(20)

,order_id,order_status,order_total_value,total_payment,payment_difference
464,8adafb3466daa5395694d3a906ff9d40,delivered,218.02,218.00,-0.02
867,bb2e64c3040ceb9b7ca2bfc602adca08,delivered,257.36,257.34,-0.02
1080,84d6d9710c8af32b5e88f2d1c14ab871,delivered,57.68,61.70,4.02
1126,74016effecaa79d592487f6a4ee47d4b,delivered,51.51,56.96,5.45
1669,239f380355f65dcb68551f07d16fc4a8,delivered,222.63,251.63,29.00
1729,4c57f545143e8865ca2347d8cba154a7,delivered,139.61,151.01,11.40
1986,6e57e23ecac1ae881286657694444267,delivered,350.41,333.91,-16.50
2277,b38b3526b8b8fdc807e8a0a42ab78573,delivered,30.06,30.19,0.13
2357,051fcda88d997d3ff86012da2a556342,delivered,56.60,51.70,-4.90
2541,8b5058499c412c6cf8d013de40e4f9d2,delivered,48.77,51.02,2.25


In [52]:
orders_enriched["payment_reconciliation_status"] = np.select(
    [
        ~orders_enriched["has_item_data"],

        ~orders_enriched["has_payment_data"],

        orders_enriched["payment_difference"].abs() <= 0.01,

        orders_enriched["payment_difference"] > 0.01,

        orders_enriched["payment_difference"] < -0.01
    ],
    [
        "Missing Item Data",
        "Missing Payment Data",
        "Matched",
        "Payment Above Order Total",
        "Payment Below Order Total"
    ],
    default="Unknown"
)

In [53]:
orders_enriched[
    "payment_reconciliation_status"
].value_counts()

payment_reconciliation_status
Matched                      98362
Missing Item Data              775
Payment Above Order Total      264
Payment Below Order Total       39
Missing Payment Data             1
Name: count, dtype: int64

In [54]:
orders_enriched[
    "payment_difference"
].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: payment_difference, dtype: float64

In [55]:
orders_enriched.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,...,freight_value,order_total_value,has_item_data,total_payment,payment_count,payment_method_count,max_installments,has_payment_data,payment_difference,payment_reconciliation_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.44,-7.11,...,8.72,38.71,True,38.71,3,2,1,True,0.0,Matched
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.78,-5.36,...,22.76,141.46,True,141.46,1,1,1,True,0.0,Matched
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.39,-17.25,...,19.22,179.12,True,179.12,1,1,3,True,0.0,Matched
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.21,-12.98,...,27.20,72.20,True,72.20,1,1,1,True,0.0,Matched
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.87,-9.24,...,8.72,28.62,True,28.62,1,1,1,True,0.0,Matched


# Adding Review to Orders

In [56]:
reviews = pd.read_csv(
    PROCESSED_PATH / "reviews_clean.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

In [57]:
reviews["has_review_comment"] = (
    reviews["review_comment_message"].notna()
)

In [58]:
review_features = reviews[
    [
        "order_id",
        "review_score",
        "has_review_comment",
        "review_creation_date",
        "review_answer_timestamp"
    ]
]

In [59]:
orders_enriched = orders_enriched.merge(
    review_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

In [60]:
orders_enriched["has_review_data"] = (
    orders_enriched["review_score"].notna()
)

In [61]:
orders_enriched["has_review_data"].value_counts()

has_review_data
True     98673
False      768
Name: count, dtype: int64

# Adding Geolocation to Customers

In [62]:
customers = pd.read_csv(
    PROCESSED_PATH / "customers_clean.csv",
    dtype={
        "customer_zip_code_prefix": "string"
    }
)

In [63]:
geolocation = pd.read_csv(
    PROCESSED_PATH / "geolocation_clean.csv",
    dtype={
        "geolocation_zip_code_prefix": "string"
    }
)

In [64]:
customers_enriched = customers.merge(
    geolocation,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

In [65]:
customers_enriched = (
    customers_enriched
    .drop(
        columns="geolocation_zip_code_prefix"
    )
    .rename(
        columns={
            "geolocation_lat": "customer_lat",
            "geolocation_lng": "customer_lng"
        }
    )
)

In [66]:
customers_enriched["has_geolocation"] = (
    customers_enriched["customer_lat"].notna() &
    customers_enriched["customer_lng"].notna()
)

In [67]:
customers_enriched.shape

(99441, 8)

In [68]:
customers_enriched[
    "has_geolocation"
].value_counts()

has_geolocation
True     99162
False      279
Name: count, dtype: int64

In [69]:
customers_enriched.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_lat,customer_lng,has_geolocation
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.499273,-47.396658,True
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,-23.728396,-46.542250,True
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,-23.531309,-46.656690,True
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,-23.500670,-46.186348,True
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.975708,-47.143140,True


# Adding Geolocation to Seller

In [70]:
sellers = pd.read_csv(
    PROCESSED_PATH / "sellers_clean.csv",
    dtype={
        "seller_zip_code_prefix": "string"
    }
)

In [71]:
sellers_enriched = sellers.merge(
    geolocation,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

In [72]:
sellers_enriched = (
    sellers_enriched
    .drop(
        columns="geolocation_zip_code_prefix"
    )
    .rename(
        columns={
            "geolocation_lat": "seller_lat",
            "geolocation_lng": "seller_lng"
        }
    )
)

In [73]:
sellers_enriched["has_geolocation"] = (
    sellers_enriched["seller_lat"].notna() &
    sellers_enriched["seller_lng"].notna()
)

In [74]:
sellers_enriched.shape

(3095, 7)

In [75]:
sellers_enriched[
    "has_geolocation"
].value_counts()

has_geolocation
True     3088
False       7
Name: count, dtype: int64

In [76]:
sellers_enriched.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng,has_geolocation
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893317,-47.060596,True
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383375,-46.948142,True
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909446,-43.180240,True
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,-23.657118,-46.612730,True
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.964546,-46.534214,True


# Making Products Details

In [77]:
products = pd.read_csv(
    PROCESSED_PATH / "products_clean.csv"
)

In [78]:
products["product_weight_kg"] = (
    products["product_weight_g"] / 1000
).round(3)

In [79]:
products["product_volume_cm3"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
).round(2)

In [80]:
products[
    [
        "product_id",
        "product_weight_g",
        "product_weight_kg",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3"
    ]
].head()

,product_id,product_weight_g,product_weight_kg,product_length_cm,product_height_cm,product_width_cm,product_volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,225.0,0.225,16.0,10.0,14.0,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,1000.0,1.000,30.0,18.0,20.0,10800.0
2,96bd76ec8810374ed1b65e291975717f,154.0,0.154,18.0,9.0,15.0,2430.0
3,cef67bcfe19066a932b7673e239eb23d,371.0,0.371,26.0,4.0,26.0,2704.0
4,9dc1a7de274444849c219cff195d0b71,625.0,0.625,20.0,17.0,13.0,4420.0


In [81]:
products[
    [
        "product_weight_kg",
        "product_volume_cm3"
    ]
].isnull().sum()

product_weight_kg     2
product_volume_cm3    2
dtype: int64

# Making Order Purchase Date

In [82]:
orders_enriched["order_purchase_date"] = (
    orders_enriched["order_purchase_timestamp"]
    .dt.normalize()
)

In [83]:
orders_enriched[
    [
        "order_purchase_timestamp",
        "order_purchase_date"
    ]
].head()

,order_purchase_timestamp,order_purchase_date
0,2017-10-02 10:56:33,2017-10-02
1,2018-07-24 20:41:37,2018-07-24
2,2018-08-08 08:38:49,2018-08-08
3,2017-11-18 19:28:06,2017-11-18
4,2018-02-13 21:18:39,2018-02-13


In [84]:
orders_enriched[
    "order_purchase_date"
].isnull().sum()

np.int64(0)

In [85]:
print("Raw orders:", len(orders))
print("Enriched orders:", len(orders_enriched))
print("Removed orders:", len(orders) - len(orders_enriched))

print("Order item rows:", len(order_items))
print(
    "Unique orders in order_items:",
    order_items["order_id"].nunique()
)

Raw orders: 99441
Enriched orders: 99441
Removed orders: 0
Order item rows: 112650
Unique orders in order_items: 98666


In [86]:
final_validation = pd.Series(
    {
        "orders_rows": len(orders_enriched),
        "duplicate_order_id": (
            orders_enriched["order_id"].duplicated().sum()
        ),

        "order_items_rows": len(order_items),
        "duplicate_order_item_key": (
            order_items
            .duplicated(["order_id", "order_item_id"])
            .sum()
        ),

        "customers_rows": len(customers_enriched),
        "duplicate_customer_id": (
            customers_enriched["customer_id"].duplicated().sum()
        ),

        "sellers_rows": len(sellers_enriched),
        "duplicate_seller_id": (
            sellers_enriched["seller_id"].duplicated().sum()
        ),

        "products_rows": len(products),
        "duplicate_product_id": (
            products["product_id"].duplicated().sum()
        ),

"payments_rows": len(payments),

"duplicate_payment_key": (
    payments
    .duplicated(["order_id", "payment_sequential"])
    .sum()
),

"missing_payment_order_id": (
    payments["order_id"]
    .isna()
    .sum()
)
    },
    name="value"
)

final_validation

orders_rows                  99441
duplicate_order_id               0
order_items_rows            112650
duplicate_order_item_key         0
customers_rows               99441
duplicate_customer_id            0
sellers_rows                  3095
duplicate_seller_id              0
products_rows                32951
duplicate_product_id             0
payments_rows               103877
duplicate_payment_key            0
missing_payment_order_id         0
Name: value, dtype: int64

In [87]:
ANALYTICS_PATH = Path("../data/analytics")

ANALYTICS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

In [88]:
orders_enriched.to_csv(
    ANALYTICS_PATH / "fact_orders.csv",
    index=False
)

order_items.to_csv(
    ANALYTICS_PATH / "fact_order_items.csv",
    index=False
)

payments.to_csv(
    ANALYTICS_PATH / "fact_payments.csv",
    index=False
)

customers_enriched.to_csv(
    ANALYTICS_PATH / "dim_customers.csv",
    index=False
)

sellers_enriched.to_csv(
    ANALYTICS_PATH / "dim_sellers.csv",
    index=False
)

products.to_csv(
    ANALYTICS_PATH / "dim_products.csv",
    index=False
)

In [89]:
print("payments rows:", len(payments))

print(
    "duplicate payment key:",
    payments.duplicated(
        ["order_id", "payment_sequential"]
    ).sum()
)

print(
    "missing order_id:",
    payments["order_id"].isna().sum()
)

payments rows: 103877
duplicate payment key: 0
missing order_id: 0
